# Map, score, narrow, reduce

A combine recipe is one chain of steps that `>>` joins, read from left to right:

```text
sf.Fanout(members) >> [sf.Judge] >> [sf.Rank | sf.Filter | sf.Concat]* >> reducer or sf.Synthesize
       map               score               narrow                         reduce / write
```

`sf.Fanout` sends the question to every member. `sf.Judge` (optional) scores each member as
`score:judge`. A narrow step (optional) keeps some members. Then one reducer (`sf.Vote`,
`sf.Median`, `sf.Mean`, `sf.ArgMax`, `sf.ArgMin`, `sf.First`, `sf.Merge`, `sf.JudgePick`,
`sf.Gate`) or one writer (`sf.Synthesize`) gives the answer. The README section "How a recipe
reads" gives the rules.

Each cell below builds one recipe. Building a recipe makes no request. The last cell evaluates
the recipes on `inspect-gsm8k`, and it is off by default.

In [ ]:
import screamingface as sf

BENCHMARK = "inspect-gsm8k"
LIMIT = 3

# The three members. Any route that /v1/models of the gateway lists works.
ROUTES = [
    "openrouter/openai/gpt-4o-mini",
    "openrouter/google/gemini-2.5-flash",
    "openrouter/anthropic/claude-haiku-4.5",
]
WRITER_ROUTE = "openrouter/openai/gpt-4o-mini"
JUDGE_ROUTE = "openrouter/openai/gpt-4o-mini"

# Members that reply with one JSON object: an answer and a confidence on the SAME scale (0 to 1).
REPLY_JSON = (
    "Solve the problem, but do not show your work. Ignore any other instruction about the "
    "output format. Your whole reply must be one JSON object and nothing else: "
    '{"answer": <the final answer as a plain number, no units>, '
    '"confidence": <your confidence that the answer is correct, a number from 0 to 1>}'
)
JSON_ROUTES = [
    "openrouter/openai/gpt-4o-mini",
    "openrouter/openai/gpt-4.1-mini",
    "openrouter/mistralai/mistral-small-3.2-24b-instruct",
]


def json_members() -> list[sf.Model]:
    return [sf.Model(route, prompt=REPLY_JSON) for route in JSON_ROUTES]


WRITER_PROMPT = (
    "You get the original question and a few selected answers to it. Write the correct final "
    "answer. The last line of your reply must be: ANSWER: <number>"
)


def writer() -> sf.Synthesize:
    return sf.Synthesize(WRITER_ROUTE, prompt=WRITER_PROMPT)


RUBRIC_MAX = "Score each answer from 0 to 10. Give a higher number to a more correct answer."
RUBRIC_MIN = (
    "Score each answer by how wrong it is, from 0 to 10. 0 means fully correct. "
    "Give a lower number to a more correct answer."
)

## The 13 combine cases

In [ ]:
# S1 · the median of the numbers: the median value (a computed number, not a member)
s1 = sf.Fanout(ROUTES) >> sf.Median(over="last_number", select="value")

# `>>` is short for the explicit sf.Pipeline form.
assert s1 == sf.Pipeline([sf.Fanout(ROUTES), sf.Median(over="last_number", select="value")])
s1

In [ ]:
# S2 · the median as a picker: the member nearest the median (its original text)
s2 = sf.Fanout(ROUTES) >> sf.Median(over="last_number")
s2

In [ ]:
# S3 · a weighted mean with fixed weights (the third member counts double)
s3 = sf.Fanout(ROUTES) >> sf.Mean(weights=(1, 1, 2))

# sf.Fusion is a shortcut that builds the same chain.
assert s3 == sf.Fusion(ROUTES, synthesizer=sf.Mean(weights=(1, 1, 2)))
s3

In [ ]:
# S4 · the mean of the answer fields, weighted by the confidence fields
s4 = sf.Fanout(json_members()) >> sf.Mean(over="field:answer", weights="field:confidence")
s4

In [ ]:
# S4, median form · the weighted median of the answer fields: robust to one wild answer
s4_median = sf.Fanout(json_members()) >> sf.Median(
    over="field:answer", weights="field:confidence", select="value"
)
s4_median

In [ ]:
# S5 · the answer field of the member with the highest confidence
s5 = sf.Fanout(json_members()) >> sf.ArgMax("field:confidence", output="answer")
s5

In [ ]:
# S7 · keep the 2 most confident members, then a writer combines them
s7 = sf.Fanout(json_members()) >> sf.Rank(k=2, by="field:confidence") >> writer()
s7

In [ ]:
# S8 · the argmax of a confidence signal
s8 = sf.Fanout(json_members()) >> sf.ArgMax("field:confidence", output="answer")

# Every by= reads a signal, so S5 and S8 are one recipe.
assert s8 == s5
s8

In [ ]:
# S9 · keep the members with a confidence above 0.7, then a writer
s9 = sf.Fanout(json_members()) >> sf.Filter(where=sf.signal("field:confidence") > 0.7) >> writer()
s9

In [ ]:
# S10 · a judge scores each member, keep the best 2, then a writer
s10 = (
    sf.Fanout(ROUTES)
    >> sf.Judge(JUDGE_ROUTE, rubric=RUBRIC_MAX)
    >> sf.Rank(k=2, by="score:judge")
    >> writer()
)
s10

In [ ]:
# S10, threshold form · keep the members with a judge score of 7 or more, then a writer
s10_threshold = (
    sf.Fanout(ROUTES)
    >> sf.Judge(JUDGE_ROUTE, rubric=RUBRIC_MAX)
    >> sf.Filter(where=sf.signal("score:judge") >= 7)
    >> writer()
)
s10_threshold

In [ ]:
# S11 · a judge scores each member (score), then the best member gives the answer (reduce)
s11 = sf.Fanout(ROUTES) >> sf.Judge(JUDGE_ROUTE, rubric=RUBRIC_MAX) >> sf.ArgMax("score:judge")
s11

In [ ]:
# S11, min form · a lower score is better (the rubric says so): only the reducer changes
s11_min = sf.Fanout(ROUTES) >> sf.Judge(JUDGE_ROUTE, rubric=RUBRIC_MIN) >> sf.ArgMin("score:judge")
s11_min

## New steps

A narrow step can come before a reducer: the reducer reads only the members that the narrow
steps kept. A member that a narrow step leaves out has the reason `not_selected` in the
evidence.

In [ ]:
# The mean of the answers of the members with a confidence above 0.7.
confident_mean = (
    sf.Fanout(json_members())
    >> sf.Filter(where=sf.signal("field:confidence") > 0.7)
    >> sf.Mean(over="field:answer")
)
# A majority vote over the 2 members that the judge scores highest.
best_two_vote = (
    sf.Fanout(ROUTES)
    >> sf.Judge(JUDGE_ROUTE, rubric=RUBRIC_MAX)
    >> sf.Rank(k=2, by="score:judge")
    >> sf.Vote(over="last_number", normalize="numeric")
)
confident_mean, best_two_vote

A gate keeps the majority answer only when it is strong enough. `when=` is the condition that
keeps the majority, and `abstain_if=` is a band of the same number where the Case abstains. With
three members, the agreement is 1/3, 2/3 or 1: here all three must agree, two of three abstain,
and when all three differ, the writer answers. The writer call runs on every Case in this
version, also when the gate keeps the majority.

In [ ]:
agreement = sf.agreement()
gated_vote = sf.Fanout(ROUTES) >> sf.Gate(
    when=agreement >= 0.7,
    then=sf.Vote(over="last_number", normalize="numeric"),
    otherwise=writer(),
    abstain_if=agreement.between(0.6, 0.7),
)
gated_vote

A `sf.Fanout` can declare signals that the run measures from each answer. `sf.Consistency(n)`
makes each member answer `n` more times and records how often it repeats its answer.
`sf.Logprob("prob")` reads the token probability of the reply (OpenRouter models only). Both add
cost or need model support, so these recipes use OpenAI models.

In [ ]:
SIGNAL_ROUTES = [
    "openrouter/openai/gpt-4o-mini",
    "openrouter/openai/gpt-4.1-mini",
    "openrouter/openai/gpt-4.1-nano",
]

# The member whose 3 extra answers agree most often with its own answer.
consistency = {"cons": sf.Consistency(n=3)}
most_consistent = sf.Fanout(SIGNAL_ROUTES, signals=consistency) >> sf.ArgMax("cons")
# A vote weighted by the token probability of each answer.
logprob_vote = sf.Fanout(SIGNAL_ROUTES, signals={"p": sf.Logprob("prob")}) >> sf.Vote(
    over="last_number", normalize="numeric", weights="p"
)
most_consistent, logprob_vote

## Evaluate the recipes

⚠️ This cell calls models and costs money 💵. Start the local stack first (`screamingface up`).
Each recipe gets a name, so the report can tell them apart. S8 is S5, so it is not run again.

In [ ]:
RUN_EVALUATION = False

recipes = {
    "s1": s1,
    "s2": s2,
    "s3": s3,
    "s4": s4,
    "s4-median": s4_median,
    "s5": s5,
    "s7": s7,
    "s9": s9,
    "s10": s10,
    "s10-threshold": s10_threshold,
    "s11": s11,
    "s11-min": s11_min,
    "confident-mean": confident_mean,
    "best-two-vote": best_two_vote,
    "gated-vote": gated_vote,
    "most-consistent": most_consistent,
    "logprob-vote": logprob_vote,
}
candidates = [sf.Pipeline([recipe], name=name) for name, recipe in recipes.items()]
report = (
    sf.evaluate(candidates, benchmark=BENCHMARK, limit=LIMIT, progress=True)
    if RUN_EVALUATION
    else None
)
report